# Lab 08 — Deploy to Production

## 兩條 Pipeline 並排

學習路徑（Labs 00–07）讓你在歷史資料上開發和驗證偵測邏輯。
目標 pipeline 把這些邏輯部署到生產環境，讓偵測自動運行。

```
Learning path (what you did in Labs)    Target deployment pipeline (automated)
─────────────────────────────           ────────────────────────────────
00 Observability stack                    exporter / SNMP poller
                                              ↓ /metrics
01 Feature engineering             →     Prometheus Recording Rules
   pandas rolling mean / std              network:traffic_mean_1h
   safe_div(errors, ucast)               network:error_rate_zscore

02 Baseline anomaly detection      →     Prometheus Alert Rules
   z-score > 3 → flag                    TrafficSurge, ErrorRateSurge
   recall / precision tuning             for: 1m (duration filter)

03 SPC                             →     Alert Rules (SPC control limits)
04 ML anomaly detection            →     Scoring service (standalone Python service)
05 Alert noise reduction           →     Alertmanager inhibition rules
06 Forecasting                     →     Forecast service + recording rules
07 RCA assistant                  →     runbook / RCA helper (human-gated)
                                              ↓
                                         Grafana
                                           ├─ real-time metrics panels
                                           ├─ Alert annotations (anomaly markers)
                                           └─ Alert list panel
                                              ↓
                                         ⚠ Human verification: confirm alerts, execute remediation
```

本 lab 完成 Lab 02 邏輯的部署：
把 z-score 閾值寫成 Prometheus recording + alert rules，
讓 Grafana 自動在時間軸上標記異常時段。


## Course architecture boundary

本課程有兩條資料路徑，請不要混在一起：

- **Actual operating path**: OS / network signals -> exporter -> Prometheus -> Grafana.
- **Python learning path**: organized telemetry CSV or previous notebook output -> Python analysis -> result CSV -> `outputs/prometheus-dropzone/current_results.csv` -> `python_results_exporter` -> Prometheus -> Grafana.

除非 cell 明確寫的是 operational deployment example，Python anomaly / forecasting / RCA notebooks 的主要輸入是 organized telemetry CSV 或前一個 notebook 的輸出，不是 Prometheus query。Grafana 也不直接讀 CSV；Grafana 查 Prometheus。


In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab08_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab08_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab08_pipeline_position.svg")


## Lab 08 概覽

### 學習目標

1. 理解 Prometheus recording rules 的預計算原理與必要性
2. 驗證告警規則已正確載入 Prometheus
3. 查看即時觸發中的 alerts，與 lab notebook 的輸出對照
4. 理解自動化部署後人類角色的轉變

### 前置條件

Lab 00 完成（Prometheus 執行中）；有 `infra/prometheus/alerts.yml`。這份檔案同時放 recording rules 與 alert rules。


## 設計主線：把 notebook 邏輯拆成可運作的生產元件

本章的實務連結是部署邊界。Notebook 適合探索，但生產環境需要 24/7 執行、可監控、可回滾、可解釋的元件。

設計部署架構時請問三個問題：

1. **哪裡適合 Prometheus**：簡單、可解釋、低延遲的 recording rules 與 alert rules 應留在 Prometheus。
2. **哪裡需要 Python service**：ML scoring、RCA context builder、LLM webhook 這類較重邏輯應獨立部署。
3. **哪裡需要人類閘道**：shutdown port、升級 incident、通知客戶等行動不能只由模型直接觸發。

設計原則：部署不是把 notebook 原封不動搬上線，而是把每段邏輯放到合適的系統位置。


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---


## Step 1 — 確認 Prometheus 已載入 alert rules

`infra/prometheus/alerts.yml` 包含：
- **Recording rules**：等同 Lab 01 計算的特徵（rolling mean、stddev、z-score）
- **Alert rules**：等同 Lab 02 的 flag 邏輯（z-score > 3 → 觸發告警）

下面確認 Prometheus 已正確載入。


In [ ]:
import urllib.request, json, pandas as pd
from pathlib import Path

def api(path: str, params: str = "") -> dict:
    url = f"http://localhost:9090/api/v1/{path}{'?' + params if params else ''}"
    with urllib.request.urlopen(url, timeout=5) as r:
        return json.loads(r.read())

# 確認 alert rules 已載入
rules = api("rules")
alert_rules = [r for g in rules["data"]["groups"]
               for r in g["rules"] if r["type"] == "alerting"]
recording_rules = [r for g in rules["data"]["groups"]
                   for r in g["rules"] if r["type"] == "recording"]

print(f"Recording rules: {len(recording_rules)}")
for r in recording_rules:
    print(f"  {r['name']}")

print(f"\nAlert rules: {len(alert_rules)}")
for r in alert_rules:
    print(f"  {r['name']}  ({r.get('state','?')})")


---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 8 分鐘，請先不要執行 cell。

---

## 📖 概念：Prometheus Recording Rules — 預計算的價值

### 為什麼需要 Recording Rules？

一個 Prometheus 查詢可能很昂貴：

```promql
# Computes in real-time on every query: rolling mean over the past 1 hour for all ports
avg_over_time(network_rrd_inoctets[1h])
```

如果你的 Grafana 儀表板有 20 個 panel，每個 panel 每 30 秒刷新一次，
Prometheus 每分鐘要重算 40 次這個查詢。在有 500 個 port 的情況下，
計算量很快就會拖垮 Prometheus 的回應速度。

**Recording rules 把昂貴的查詢預先計算，把結果存成新的 metric。**

```yaml
# infra/prometheus/alerts.yml
groups:
  - name: network_baselines
    rules:
      - record: network:traffic_mean_1h
        expr: avg_over_time(network_rrd_inoctets[1h])
        # Prometheus evaluates this every 15 s and stores the result as network:traffic_mean_1h
```

之後的查詢直接讀 `network:traffic_mean_1h`，不需要重算。

### Recording rules 的兩個用途

| 用途 | 說明 | 範例 |
|---|---|---|
| **效能優化** | 把複雜計算預算，加速 Grafana 查詢 | rolling mean、rolling std |
| **告警前置條件** | 告警規則引用 recording rule，避免重複計算 | z-score 計算結果作為告警輸入 |

### 與 Lab 01-05 的對應關係

你在 lab 裡用 pandas 計算的特徵，和 recording rules 的 PromQL 是同一件事：

| Lab 01 pandas 計算 | 對應的 Recording Rule |
|---|---|
| `df["error_rate"] = df["INERRORS"] / df["INUCASTPKTS"]` | `network:error_rate` (PromQL: `network_rrd_inerrors / network_rrd_inucastpkts`) |
| `df["traffic_total_mean_1h"]` | `network:traffic_mean_1h` |
| `df["traffic_zscore"]` | `network:traffic_zscore` |

**這就是「兩條軌道整合」的具體實現**：lab 驗證了邏輯是正確的，然後把同樣的邏輯翻譯成 PromQL 部署到 Prometheus。

### 適用條件

Recording rules 最適合：「固定計算邏輯、需要高頻查詢、計算成本高」的場景。
複雜的 ML 推論（如 Isolation Forest）無法用 PromQL 表達，仍需要外部 Python service 產生數值結果，再由 exporter 暴露 `/metrics` 讓 Prometheus scrape。


## Step 2 — 查詢 recording rules 的計算結果

Recording rules 每 5 秒預先計算好，存成新的 metric。
這讓 alert rule 和 Grafana 都能直接用，不需要每次重新計算。


In [ ]:
def promql(expr: str) -> pd.DataFrame:
    url = "http://localhost:9090/api/v1/query?query=" + urllib.request.quote(expr)
    with urllib.request.urlopen(url, timeout=5) as r:
        result = json.loads(r.read())
    rows = []
    for item in result["data"]["result"]:
        row = {**item["metric"]}
        row["value"] = float(item["value"][1])
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

import urllib.parse
def promql(expr: str) -> pd.DataFrame:
    url = "http://localhost:9090/api/v1/query?query=" + urllib.parse.quote(expr)
    with urllib.request.urlopen(url, timeout=5) as r:
        result = json.loads(r.read())
    rows = []
    for item in result["data"]["result"]:
        row = {**item["metric"]}
        row["value"] = float(item["value"][1])
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

# 各 port 目前的 z-score（等同 Lab 02 的即時版本）
df_z = promql("network:traffic_zscore")
if not df_z.empty:
    print("目前各 port traffic z-score：")
    print(df_z[["port_id", "device_id", "value"]].rename(
        columns={"value": "z_score"}).to_string(index=False))
    print()
    print("z_score > 3 → Prometheus 會觸發 TrafficSurge alert")
    print("z_score < -3 → Prometheus 會觸發 TrafficDrop alert")
else:
    print("z-score 尚未計算（等待 Prometheus 累積 1 小時歷史，或 exporter 尚未啟動）")


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---


## Step 3 — 查看目前觸發中的 alerts

每個 alert 都是 Lab 02 裡一個 flag = 1 的等價物，
差別在這裡是即時觸發，不需要人工執行 cell。


In [ ]:
alerts = api("alerts")
firing = [a for a in alerts["data"]["alerts"] if a["state"] == "firing"]

if firing:
    print(f"目前觸發中的 alerts：{len(firing)} 個\n")
    for a in firing:
        labels = a["labels"]
        ann = a["annotations"]
        print(f"  🔴 {labels.get('alertname')}  [{labels.get('severity')}]")
        print(f"     Port: {labels.get('port_id')} | {labels.get('device_id')}")
        print(f"     {ann.get('summary','')}")
        print()
else:
    print("目前沒有觸發中的 alerts（正常狀態，或 exporter 尚未運行enough history）")
    print()
    print("等待 exporter 回放到有事件的時段後，alert 會自動出現。")
    print("全部 5 種事件：traffic_surge, error_burst, discard_spike,")
    print("               traffic_drop, broadcast_storm")


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---


## Step 4 — Grafana Alert Annotations

Prometheus alerts 觸發後，Grafana 可以從 Prometheus 查 `ALERTS` series，並在 dashboard 上標記事件。

### 在 Grafana Local 設定 Alert Annotations

1. 開啟 `http://localhost:3000` -> **Dashboards** -> **Network Interface Monitoring** 儀表板
2. 右上角 **Dashboard settings** -> **Annotations**
3. 新增 annotation source：
   - Data source: **Prometheus**
   - Query: `ALERTS{alertname=~"TrafficSurge|TrafficDrop|ErrorRateSurge|DiscardRateSurge|BroadcastStorm"}`
   - Color: orange or red
4. 儲存並回到儀表板

這樣每次 alert 觸發，時間軸上就會出現垂直標記線。這是 Lab 02 的 anomaly flag 在 production dashboard 裡的視覺化呈現。

本課程不要求任何外部 dashboard credential。學員只需要確認 Prometheus 有 `ALERTS` metrics，然後在 Grafana UI 手動選 Prometheus annotation source。


In [ ]:
# Verify that Prometheus exposes ALERTS metrics for Grafana annotations.
# This checks the data source Grafana will query. It stays inside the local Prometheus path.

alert_series = promql('ALERTS')
if len(alert_series):
    print(f"✅ Prometheus ALERTS series available: {len(alert_series)} rows")
    display(alert_series.head())
else:
    print("ℹ No ALERTS series returned right now.")
    print("This can mean no alert is active, exporter replay has not reached an event window, or alert rules need reload.")
    print("Grafana annotation setup can still be completed; it will show markers when ALERTS appears.")


In [ ]:
print("Grafana annotation path:")
print("  Prometheus alert rules -> ALERTS metric -> Grafana annotation query")
print()
print("Manual Grafana query to use:")
print('  ALERTS{alertname=~"TrafficSurge|TrafficDrop|ErrorRateSurge|DiscardRateSurge|BroadcastStorm"}')
print()
print("No external dashboard credential is required for the course workflow.")


---
💬 **討論暫停 ▸ 全班討論** — 停止執行，分享你的觀察。

---


## ⚠ 人類驗證點 #8 — 自動化部署後，人類的角色更重要，不是更少

Pipeline 自動化之後：
- Prometheus 24/7 計算 z-score，不需要你執行 lab
- Grafana 即時顯示異常標記
- Alerts 自動觸發

**但這不代表可以不看 Grafana。這代表你需要更有紀律地審查它。**

| 自動化做了什麼 | 人類還需要做什麼 |
|---|---|
| 依閾值標記所有超出 baseline 的點 | 判斷哪些是真實事件、哪些是計畫維護或業務高峰 |
| 觸發 alert | 決定要不要 escalate、execute runbook |
| 計算 z-score | 定期重新評估閾值是否仍然適合（網路行為會隨業務演變）|
| 比對歷史 baseline | 知道這個告警以前出現過嗎？結果如何？|

**一個沒有人審查的自動化告警系統，很快會被當作背景雜音忽略。**
這是 alert fatigue 的根本原因——不是告警太多，而是告警沒有被信任。

Lab 05（告警降噪）是這個問題的下一步答案。

---

## 整合完成：兩條軌道收斂

```
What you did in Labs              Automated Production Pipeline
────────────────────             ──────────────────────────────
Lab 01 feature engineering ───→    infra/prometheus/alerts.yml recording rules
Lab 02 z-score thresholds ────→    infra/prometheus/alerts.yml alert rules
                                          ↓
                                 Grafana annotations (Prometheus annotation query)
                                          ↓
                                 ⚠ You: verify, decide, feed back into Labs 03–07
```

Lab 03–07 的邏輯可以用相同方式逐步部署，每個 lab 完成後
把閾值或規則寫進 `infra/prometheus/alerts.yml`，Prometheus 產生 `ALERTS` series 後，Grafana annotation query 會顯示事件。


### 如何判斷自動化部署是否成功（判斷標準）

1. Step 1 的 alert rules 查詢返回非空結果，且規則名稱與 `alerts.yml` 一致
2. Step 2 的 recording rules 返回數值，且數值與 Lab 01 的 pandas 計算結果在同一量級
3. Step 3 的 active alerts 在已知事件時段（依 exporter replay Time）正確觸發

**讓你重新考慮的信號**

- Recording rule 的值是 `NaN` → PromQL 表達式有錯或 metric 名稱拼錯
- Alert 一直維持在 `pending` 但不轉成 `firing` → `for` duration可能比測試窗口長
- Alert 完全不出現 → `expr` 的閾值可能太高，或資料根本沒有超標


---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：告警規則的 `for` duration

在 `infra/prometheus/alerts.yml` 中，找到一個告警規則的 `for` 欄位（例如 `for: 5m`），
改成 `for: 2m`，重新 reload Prometheus config（Step 1 的 reload cell），
然後觀察 Step 3 的 active alerts 是否更快出現。

```yaml
# Before modification
- alert: HighErrorRate
  expr: network:error_rate_zscore > 3
  for: 5m          # alert only after 5 consecutive minutes above threshold

# After modification
- alert: HighErrorRate
  expr: network:error_rate_zscore > 3
  for: 2m          # alert after 2 consecutive minutes above threshold
```

**如何判斷這個改動是否合理**

- 縮短 `for` duration後，如果 pending alerts 數量明顯增加但多數又很快消失，
  說明原始的 5 分鐘是合理的防抖動設計。
- 如果 pending alerts 穩定維持，說明 2 分鐘也夠用。

**讓你重新考慮的信號**

- 縮短後出現大量 flapping（告警不斷觸發再消失）→ 短暫的流量波動被誤當成事件
- 縮短後完全沒有差別 → 事件持續時間遠超 for 的設定，調整無意義

---

> 「Lab 02 我們把 Z_THRESH 設成 3.0；current Prometheus 的告警規則裡是多少？兩個地方的值是否一致？如果不一致，哪個才算？」

> 「自動化部署之後，你覺得 NOC 工程師的工作Time分配會怎麼變化？哪些工作增加了，哪些工作減少了？」

> 「如果 Prometheus 本身掛掉了，你怎麼知道？（提示：誰監控監控系統？）」


## Optional: update Grafana with deployment outputs

本章重點是把 notebook 邏輯拆成 production components。若某個 Python service 產生新的 CSV 結果，可以複製到：

```bash
cp <your-result.csv> outputs/prometheus-dropzone/current_results.csv
```

保持 `python infra/python_results_exporter.py` 執行後，Grafana 就能用 `aiops_python_result` 查到結果。完整流程見 `labs/getting-started/05-prometheus-dropzone.md`。
